# 07 — Eksperimen 10 Machine Learning

**10 model ML × 5 horizon × 2 setup = 100 run**: LightGBM, XGBoost, Random Forest, CatBoost, Gradient Boosting, SVR, Extra Trees, HistGradientBoosting, AdaBoost, ElasticNet.

Aturan anti-kebocoran:
- split kronologis 70/10/20; early stopping boosting (LightGBM, XGBoost, CatBoost) memakai baris validasi **ber-embargo**; model non-boosting tidak memakai validasi;
- fitur lag/rolling hanya memakai informasi sampai hari origin; makro sudah lag-1;
- scaler SVR/ElasticNet fit hanya pada train; uji hanya dipakai sekali untuk evaluasi final;
- hiperparameter **tidak** dituning pada uji (nilai tetap di `src/pipeline.py`/`src/config.py`).

**Revisi konfigurasi `ml-r2` (SVR).** Versi lama memakai `epsilon = 0,02` pada target mentah; untuk h=1 simpangan baku return hanya ≈0,7%, sehingga seluruh target jatuh di dalam tabung-ε dan SVR hanya menghasilkan konstanta. Versi ini memakai target terstandardisasi (default sklearn `C=1, ε=0,1` pada satuan simpangan baku). Karena itu **seluruh ML dijalankan ulang otomatis** (± 6 menit) — hasil ML lama tanpa penanda `rev` dianggap basi. DL dan MOSSA **tidak** terpengaruh (protokol tidak berubah).

Drift (rata-rata return h-hari pada train) disimpan sebagai benchmark diagnostik; ia **bukan** salah satu dari 21 model.

In [1]:
import sys, os, json, time, warnings
sys.path.insert(0, os.path.join('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import (OUT, HORIZONS, SETUPS, ML_MODELS, DL_MODELS, ALL_MODELS, LANG, PROTOCOL, ML_REV, QUICK)
from src import analysis as A, plots as P, stats as S
from src.runners import load_data, data_fingerprint

P.set_lang(LANG)
RES = os.path.join(OUT, 'results')
RUNS = os.path.join(RES, 'runs')
FIGS = os.path.join(OUT, 'figures')
for _d in (RES, RUNS, FIGS):
    os.makedirs(_d, exist_ok=True)

df, p, exo = load_data('..')
DATA_HASH = data_fingerprint(df)
print(f'Data: {len(p)} hari kerja, {p.index[0].date()} s.d. {p.index[-1].date()} | sidik jari {DATA_HASH}'
      + (' | MODE QUICK (hasil hanya uji kelancaran)' if QUICK else ''))


def simpan(fig, nama):
    """Simpan figur (PNG + PDF) ke outputs/figures lalu tampilkan."""
    P.save_fig(fig, nama, FIGS)
    plt.show()
    plt.close(fig)

from src import runners as R
from src.pipeline import build_lgb_features, drift_h
from src.config import LGB_PARAMS
FORCE_RERUN = False

F = {'univariat': build_lgb_features(p), 'multivariat': build_lgb_features(p, exo)}
assert len(ML_MODELS) == 10
print('ML:', ML_MODELS)
print('Fitur:', F['univariat'].shape[1], 'univariat |', F['multivariat'].shape[1], 'multivariat')
print('LightGBM params:', LGB_PARAMS, '| revisi ML:', ML_REV)
print('Total:', len(ML_MODELS) * len(HORIZONS) * len(SETUPS), 'run')

Data: 4325 hari kerja, 2010-01-05 s.d. 2026-08-03 | sidik jari 7aea8db80d44
ML: ['lightgbm', 'xgboost', 'rf', 'catboost', 'gbm', 'svr', 'extra_trees', 'hist_gbm', 'adaboost', 'elasticnet']
Fitur: 15 univariat | 22 multivariat
LightGBM params: {'learning_rate': 0.02, 'num_leaves': 15, 'min_child_samples': 5} | revisi ML: ml-r2
Total: 100 run


In [2]:
def run_fn(setup, model, h):
    return R.run_ml(p, exo if setup == 'multivariat' else None, setup, model, h, F=F[setup], rev=ML_REV, data_hash=DATA_HASH)


tasks = [(s, m, h) for s in SETUPS for m in ML_MODELS for h in HORIZONS]
cnt = R.run_grid(tasks, run_fn, RUNS, rev=ML_REV, data_hash=DATA_HASH, force=FORCE_RERUN)
assert cnt['gagal'] == 0, f"Ada run gagal: {cnt['daftar_gagal']}"

RUN  univariat/lightgbm/h1 ... Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[11]	valid_0's rmse: 0.00655248
OK MAE=13,267 (1.2s)
RUN  univariat/lightgbm/h5 ... Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[11]	valid_0's rmse: 0.0130926
OK MAE=29,976 (0.2s)
RUN  univariat/lightgbm/h10 ... Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.0186202
OK MAE=42,637 (0.2s)
RUN  univariat/lightgbm/h20 ... Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[7]	valid_0's rmse: 0.024538
OK MAE=62,828 (0.3s)
RUN  univariat/lightgbm/h30 ... Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's rmse: 0.0283184
OK MAE=83,094 (0.3s)
RUN  univariat/xgboost/h1 ... OK MAE=13,317 (3.3s)
RUN  univariat/xgboost/h5 ... OK MAE=29,953 (0.8s)
RUN

### Benchmark Drift — bukan bagian dari 21 model

Drift memprediksi return h-hari = rata-rata return h-hari pada **train** (konstan), tanpa parameter yang dilatih. Ia dihitung di sini supaya kita tahu apakah model mengalahkan prediktor tren yang paling sederhana. (Naif/random-walk murni tidak dipakai sebagai benchmark utama karena pada harga yang naik terus ia sistematis tertinggal; Drift lebih menantang untuk komoditas yang trending.)

In [3]:
bench = []
for h in HORIZONS:
    metrics, rows, y_true, y_pred, mu = drift_h(p, h)
    bench.append({'setup': 'benchmark', 'model': 'drift', 'horizon': h, **metrics, 'drift_mu': mu, 'protocol': PROTOCOL})
bench = pd.DataFrame(bench)
bench.to_csv(os.path.join(RES, 'drift_benchmark.csv'), index=False)
display(bench.round(4))

,setup,model,horizon,MAE,RMSE,sMAPE,MAPE,drift_mu,protocol
0,benchmark,drift,1,13247.1165,21815.2571,0.6915,0.6912,0.0003,v9-21model-dpformer-bao-aligned-honest-mossa-s...
1,benchmark,drift,5,30088.7333,46740.0689,1.5720,1.5650,0.0014,v9-21model-dpformer-bao-aligned-honest-mossa-s...
2,benchmark,drift,10,42684.6932,67640.4155,2.2364,2.2146,0.0029,v9-21model-dpformer-bao-aligned-honest-mossa-s...
3,benchmark,drift,20,62952.9907,97801.2385,3.2353,3.1735,0.0060,v9-21model-dpformer-bao-aligned-honest-mossa-s...
4,benchmark,drift,30,83313.3884,127185.4048,4.2087,4.0945,0.0091,v9-21model-dpformer-bao-aligned-honest-mossa-s...


In [4]:
hasil = R.read_runs(RUNS, SETUPS, ML_MODELS, HORIZONS, rev=ML_REV, data_hash=DATA_HASH)
expected = len(ML_MODELS) * len(HORIZONS) * len(SETUPS)
print(f'Hasil ML valid: {len(hasil)} / {expected}')
assert len(hasil) == expected, 'Masih ada kombinasi ML yang belum selesai.'
display(hasil.pivot_table(index='model', columns=['setup', 'horizon'], values='MAE').round(0))

Hasil ML valid: 100 / 100


setup       multivariat                                     univariat  \
horizon              1        5        10       20       30        1    
model                                                                   
adaboost        13246.0  29649.0  41033.0  59958.0  82061.0   13222.0   
catboost        13368.0  30111.0  42659.0  62920.0  83198.0   13356.0   
elasticnet      13316.0  30021.0  42084.0  61659.0  83173.0   13271.0   
extra_trees     13605.0  30077.0  42101.0  61871.0  83106.0   13812.0   
gbm             14117.0  30606.0  43153.0  62620.0  84654.0   14512.0   
hist_gbm        14075.0  31457.0  43377.0  63460.0  85662.0   14537.0   
lightgbm        13190.0  30189.0  42264.0  62214.0  83073.0   13267.0   
rf              13493.0  30085.0  41818.0  61052.0  81608.0   13609.0   
svr             13636.0  30283.0  43726.0  66349.0  87807.0   13641.0   
xgboost         13379.0  29948.0  42437.0  61544.0  83175.0   13317.0   

setup                                            
horizon           5        10       20       30  
model                                            
adaboost     29915.0  41069.0  58692.0  82467.0  
catboost     30089.0  42651.0  62870.0  83217.0  
elasticnet   29925.0  42073.0  61664.0  82995.0  
extra_trees  30235.0  41424.0  61104.0  81681.0  
gbm          31018.0  43692.0  61563.0  85214.0  
hist_gbm     31493.0  43116.0  63494.0  84563.0  
lightgbm     29976.0  42637.0  62828.0  83094.0  
rf           30166.0  41453.0  60586.0  80637.0  
svr          29708.0  43118.0  67060.0  87057.0  
xgboost      29953.0  42573.0  62725.0  83175.0